# V2 Model Evaluation

**Goal:** Test dependency predictions while keeping certificate decisions independent.

*Experimental demo · results first, technical evidence below.*

In [1]:
from IPython.utils.capture import capture_output

EXPERIMENT_CELLS = [{'id': '1de6b87f', 'source': 'import os, sys, json, html, tempfile, importlib.util, math\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom unittest.mock import Mock, patch\nfrom IPython.display import display, HTML, Markdown\n\nREPO_LOCATION = os.environ.get("CERT_RECOVERY_REPO", "")\nV2_LOCATION = os.environ.get("CERT_RECOVERY_V2_RUN", "")\ncandidates = [Path(REPO_LOCATION).expanduser()] if REPO_LOCATION else [\n    Path.cwd(), *Path.cwd().parents, Path.cwd() / "cap-stone-current"]\nROOT = next((p.resolve() for p in candidates\n             if (p / "src/cert_recovery/controlled_study.py").is_file()), None)\nif ROOT is None:\n    raise RuntimeError("Set CERT_RECOVERY_REPO to your current repository checkout, then rerun.")\nsys.path.insert(0, str(ROOT / "src"))\nfrom cert_recovery.data_pipeline import clean_rows, load_verified_splits, write_json\nfrom cert_recovery.model_pipeline import _infer, _prediction_report\nfrom cert_recovery.controlled_study import verify_frozen, execute_stage\nfrom cert_recovery.pinned_smoke import sha256_file\nimport cert_recovery.controlled_study as study\n\ndef table(rows, columns=None):\n    columns = columns or list(rows[0]) if rows else columns or []\n    def escaped(value):\n        if isinstance(value, (dict, list)):\n            value = json.dumps(value, ensure_ascii=False)\n        return html.escape(str(value)).replace("\\n", "<br>")\n    body = \'\'.join(\'<tr>\' + \'\'.join(\'<td>\' + escaped(r.get(c, \'\')) + \'</td>\' for c in columns)\n                   + \'</tr>\' for r in rows)\n    display(HTML(\'<style>table.demo{border-collapse:collapse;font:14px system-ui;max-width:100%}\'\n                 \'table.demo th,table.demo td{border:1px solid #ccd4de;padding:9px;text-align:left;\'\n                 \'vertical-align:top;overflow-wrap:anywhere}table.demo th{background:#edf2f8}</style>\'\n                 \'<table class="demo"><tr>\' + \'\'.join(\'<th>\' + escaped(c) + \'</th>\' for c in columns)\n                 + \'</tr>\' + body + \'</table>\'))\n\ndef observed_error(action):\n    try:\n        action()\n    except Exception as error:\n        return {"type": type(error).__name__, "message": str(error)}\n    return {"type": "NO_ERROR", "message": "No rejection occurred"}\n\nevidence, before_after = [], []\ntable([{"Source": "Verified current checkout + existing V2 test fixtures",\n        "Training": "Not executed", "Model replay": "Local bundle opt-in only",\n        "V3": "Not imported or executed"}])'}, {'id': 'b42747a6', 'source': 'fixture = ROOT / "tests/remote/fixtures/case_003_recorded_response.json"\nhistorical = json.loads(fixture.read_text(encoding="utf-8"))\nrecord = historical["record"]\nnormal = record["normalized_output"]\ntable([{"Evidence": historical["fixture_type"],\n        "Input": record["input"], "Expected NLI": record["expected_label"],\n        "Actual NLI": normal["label"], "Reported confidence": normal["confidence_text"],\n        "Matches expectation": record["matches_expected_label"],\n        "Loaded model identity": record["model_identity_status"]}])\nassert record["expected_label"] == "neutral" and normal["label"] == "contradiction"\nevidence.append({"Area": "Historical semantic failure", "Experiment": "Read recorded case 003",\n                 "Actual observation": "Neutral annotation; contradiction at 99.57%; no fresh inference"})\ndisplay(Markdown("**Observation:** incorrect, high-confidence historical NLI response. "\n                 "The original binary checkpoint is not available here; its historical predictions "\n                 "cannot be independently reproduced in this environment."))'}, {'id': 'fcd310b6', 'source': 'fixture_spec = importlib.util.spec_from_file_location(\n    "mentor_v2_existing_fixture", ROOT / "tests/test_v2_execution.py")\nfixture_module = importlib.util.module_from_spec(fixture_spec)\nfixture_spec.loader.exec_module(fixture_module)\nscratch = tempfile.TemporaryDirectory(prefix="v2_mentor_demo_")\nscratch_root = Path(scratch.name)\ndef fresh_fixture(name):\n    folder = scratch_root / name\n    folder.mkdir()\n    return fixture_module.frozen_run.__wrapped__(folder)\ntable([{"Demonstration backend": "Existing frozen_run fixture",\n        "Rows": "6 synthetic rows; separate train/validation/test groups",\n        "Revision field": "Synthetic placeholder; never loaded",\n        "Writes": "Disposable temporary directory only"}])'}, {'id': 'd4f7d39d', 'source': 'raw_pair = {"pair_id": "mentor-clean", "world_id": "mentor-world",\n            "premise": "  Q-only:\\n source Cafe\\u0301 is not 17.  All sources have no supports. ",\n            "hypothesis": " Q uses Cafe\\u0301 only.\\tAll other sources are excluded. ",\n            "label": 1, "source": "Explicitly synthetic text-handling demonstration"}\ncleaned, cleaning = clean_rows([raw_pair])\naccepted = cleaned[0]\nexpected_text = "Q-only: source Café is not 17. All sources have no supports."\nassert accepted["premise"] == expected_text\nassert accepted["hypothesis"] == "Q uses Café only. All other sources are excluded."\ntable([{"Field": field, "Before": repr(raw_pair[field]), "Expected": expected,\n        "Actual": accepted[field], "Match": accepted[field] == expected}\n       for field, expected in [("premise", expected_text),\n                               ("hypothesis", "Q uses Café only. All other sources are excluded.")]])\nevidence.append({"Area": "Input handling", "Experiment": "Real clean_rows on a synthetic pair",\n                 "Actual observation": "Whitespace/NFC cleaned; scope, negation and 17 preserved"})\nbefore_after.append({"Test case": "Complete scoped input", "Before": "Raw inconsistent whitespace/Unicode",\n                     "V2": "Exact expected normalized text", "Difference": "Formatting only; task text preserved"})\ndisplay(Markdown("**Observation:** correct text handling for this example. No model accuracy claim."))'}, {'id': 'cac14d07', 'source': 'synthetic_probability = 0.30\nillustrative_threshold = 0.20\ndecision_report = _prediction_report([accepted], [synthetic_probability], illustrative_threshold, 10)\ndecision = decision_report["predictions"][0]\nassert decision["prediction"] == 1 and decision["predicted_label_probability"] == 0.30\ntable([{"Input origin": "Synthetic probability; NOT model output",\n        "P(depends_on)": decision["probability"], "Threshold": decision_report["threshold"],\n        "Expected / actual prediction": f"1 / {decision[\'prediction\']}",\n        "All binary probabilities": decision["class_probabilities"],\n        "Predicted-label probability": decision["predicted_label_probability"],\n        "Illustrative 0.50 decision": int(synthetic_probability >= 0.50)}])\nbad_probability = observed_error(lambda: _prediction_report([accepted], [float(\'nan\')], 0.20, 10))\nassert bad_probability["type"] == "ValueError"\ntable([{"Malformed input": "NaN probability", "Actual rejection": bad_probability}])\nevidence.append({"Area": "Threshold behavior", "Experiment": "Real reporter; injected probability 0.30",\n                 "Actual observation": "Threshold .20 yields label 1 with probability .30; NaN rejected"})\nbefore_after.append({"Test case": "Threshold decision (synthetic)", "Before": "Illustrative .50 cutoff: label 0",\n                     "V2": "Illustrative frozen .20 cutoff: label 1",\n                     "Difference": "Shows the actual decision rule; not a measured model improvement"})\ndisplay(Markdown("**Observation:** decision logic behaves as expected. A lower threshold trades "\n                 "missed dependencies against false alarms; this example does not establish the best threshold."))'}, {'id': 'f7354d62', 'source': 'integrity_run = fresh_fixture("integrity")\nverified_config, verified_protocol = verify_frozen(integrity_run)\nvalidation_path = integrity_run / "splits/validation.jsonl"\nuntouched_bytes = validation_path.read_bytes()\nvalidation_rows = [json.loads(line) for line in validation_path.read_text().splitlines()]\nvalidation_rows[0]["premise"] = "Altered demonstration text"\nvalidation_path.write_text(\'\'.join(json.dumps(r) + "\\n" for r in validation_rows))\nintegrity_error = observed_error(lambda: verify_frozen(integrity_run))\nassert integrity_error == {"type": "ValueError", "message": "Empty or altered split: validation"}\nvalidation_path.write_bytes(untouched_bytes)\nverify_frozen(integrity_run)\ntable([{"Bundle state": "Untouched", "Expected": "Accept", "Actual": "Accepted by verify_frozen"},\n       {"Bundle state": "Validation text changed", "Expected": "Reject", "Actual": integrity_error},\n       {"Bundle state": "Original bytes restored", "Expected": "Accept", "Actual": "Accepted again"}])\nevidence.append({"Area": "Artifact integrity", "Experiment": "Alter disposable validation text",\n                 "Actual observation": "ValueError: Empty or altered split: validation"})\nbefore_after.append({"Test case": "Artifact integrity", "Before": "Valid frozen copy accepted",\n                     "V2": "Changed copy rejected", "Difference": "Change detected; restored bytes accepted"})\ndisplay(Markdown("**Observation:** the integrity check catches this change. "\n                 "It does not prove the annotations themselves are correct."))'}, {'id': 'aca5151b', 'source': 'ordering_run = fresh_fixture("ordering")\nmanifest_before = (ordering_run / "manifest.json").read_bytes()\nordering_error = observed_error(lambda: execute_stage(ordering_run, "comparison"))\nattempts = len(list(ordering_run.glob("attempt_*")))\nassert ordering_error["message"] == "Successful selected training required"\nassert attempts == 0 and (ordering_run / "manifest.json").read_bytes() == manifest_before\ntable([{"Input": "comparison with no successful training", "Expected": "Reject before attempt",\n        "Actual": ordering_error, "Attempts created": attempts, "Manifest changed": False}])\nevidence.append({"Area": "Execution control", "Experiment": "Comparison before training",\n                 "Actual observation": "Rejected; zero attempts; manifest unchanged"})\ndisplay(Markdown("**Observation:** the invalid order was blocked without running a model."))'}, {'id': 'bb43c9d0', 'source': 'failure_run = fresh_fixture("failure")\nhardware_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: False))\nwith patch.dict(sys.modules, {"torch": hardware_adapter}), patch.object(\n        study, "run_finetuning", side_effect=RuntimeError("Controlled mentor-demo backend failure")) as backend:\n    failure_error = observed_error(lambda: execute_stage(failure_run, "training"))\n    calls = backend.call_count\nfailure_manifest = json.loads((failure_run / "manifest.json").read_text())\nfailure_state = failure_manifest["stages"]["training"]\npersisted = json.loads((Path(failure_state["attempt"]) / "status.json").read_text())\nassert persisted == failure_state and failure_state["status"] == "ERROR" and calls == 1\nretry_error = observed_error(lambda: execute_stage(failure_run, "training"))\nassert retry_error["type"] == "FileExistsError"\ntable([{"Event": "Injected backend exception", "Expected": "ERROR evidence retained",\n        "Actual": failure_error, "Persisted status": persisted["status"],\n        "Attempt/manifest agree": persisted == failure_state},\n       {"Event": "Retry same stage", "Expected": "Reject overwrite/retry",\n        "Actual": retry_error, "Persisted status": failure_state["status"],\n        "Attempt/manifest agree": True}])\ngpu_run = fresh_fixture("simulated_gpu")\ngpu_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: True))\nwith patch.dict(sys.modules, {"torch": gpu_adapter}), patch.object(study, "run_finetuning") as forbidden_backend:\n    gpu_error = observed_error(lambda: execute_stage(gpu_run, "training"))\n    backend_called = forbidden_backend.called\ngpu_state = json.loads((gpu_run / "manifest.json").read_text())["stages"]["training"]\nassert gpu_error["message"] == "This frozen experiment is local CPU only" and not backend_called\nassert gpu_state["status"] == "ERROR"\ntable([{"Simulation": "GPU available", "Expected": "Block this CPU-only protocol",\n        "Actual": gpu_error, "Training callback called": backend_called,\n        "Persisted status": gpu_state["status"]}])\nevidence.extend([\n    {"Area": "Failure handling", "Experiment": "Synthetic backend exception + retry",\n     "Actual observation": "ERROR retained in both records; retry rejected"},\n    {"Area": "CPU-only guard", "Experiment": "Simulated GPU availability",\n     "Actual observation": "Rejected and recorded ERROR; training callback not called"}])\ndisplay(Markdown("**Observation:** failures remain visible. This protocol-specific CPU guard "\n                 "does not imply that DeBERTa generally cannot run on a GPU."))'}, {'id': '295197ad', 'source': 'replay_status, replay_results, replay_report = "NOT RUN", [], None\nreplay_error, actual_config, actual_selection = None, None, None\nif not V2_LOCATION:\n    replay_reason = "Private V2 bundle not configured; no model outputs generated."\nelse:\n    try:\n        run = Path(V2_LOCATION).expanduser().resolve()\n        actual_config, protocol = verify_frozen(run)\n        actual_selection = json.loads((run / "checkpoints/selection.json").read_text())\n        checkpoint = run / "checkpoints/best"\n        if sha256_file(checkpoint / "model.safetensors") != actual_selection["checkpoint_sha256"]:\n            raise ValueError("V2 checkpoint hash mismatch")\n        for filename, digest in actual_selection["checkpoint_file_hashes"].items():\n            if sha256_file(checkpoint / filename) != digest:\n                raise ValueError(f"Checkpoint file changed: {filename}")\n        splits, _ = load_verified_splits(actual_config)\n        saved = json.loads((run / "comparisons/validation_candidate_v2.json").read_text())\n        saved_by_id = {r["pair_id"]: r for r in saved["predictions"]}\n        for row in splits["validation"]:\n            recorded = saved_by_id[row["pair_id"]]\n            if any(recorded[k] != row[k] for k in ("premise", "hypothesis", "label")):\n                raise ValueError("Saved prediction input/label differs from verified validation row")\n        selected_ids = []\n        for label in (1, 0):\n            eligible = [r for r in saved["predictions"] if r["label"] == label and r["prediction"] == label]\n            if eligible:\n                selected_ids.append(eligible[0]["pair_id"])\n        incorrect = [r for r in saved["predictions"] if r["prediction"] != r["label"]]\n        if incorrect:\n            selected_ids.append(max(incorrect, key=lambda r: r["probability"] if r["prediction"] else 1-r["probability"])["pair_id"])\n        for row in splits["validation"]:\n            if len(selected_ids) >= 3:\n                break\n            if row["pair_id"] not in selected_ids:\n                selected_ids.append(row["pair_id"])\n        verified_by_id = {r["pair_id"]: r for r in splits["validation"]}\n        replay_rows = [verified_by_id[p] for p in selected_ids]\n        os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")\n        import torch\n        from transformers import AutoTokenizer, AutoModelForSequenceClassification\n        tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=False)\n        model = AutoModelForSequenceClassification.from_pretrained(\n            checkpoint, local_files_only=True, trust_remote_code=False, use_safetensors=True)\n        if model.config.id2label != {0: "independent", 1: "depends_on"}:\n            raise ValueError("Unexpected V2 binary label mapping")\n        for row in replay_rows:\n            if len(tokenizer(row["premise"], row["hypothesis"], truncation=False)["input_ids"]) > actual_config["model"]["max_length"]:\n                raise ValueError("Replay would truncate complete task context")\n        capture = []\n        with patch.object(torch.cuda, "is_available", return_value=False):\n            probabilities = _infer(actual_config, replay_rows, model, tokenizer, 1, capture)\n        replay_report = _prediction_report(replay_rows, probabilities, actual_selection["threshold"],\n                                            actual_config["model"]["calibration_bins"], capture)\n        for actual in replay_report["predictions"]:\n            recorded = saved_by_id[actual["pair_id"]]\n            replay_results.append({"Pair ID": actual["pair_id"],\n                "Annotation expected": actual["label"], "Saved V2 prediction": recorded["prediction"],\n                "Actual V2 prediction": actual["prediction"], "P(depends_on)": actual["probability"],\n                "Frozen threshold": actual_selection["threshold"], "Logits": actual["raw_logits"],\n                "All probabilities": actual["original_model_class_probabilities"],\n                "Matches annotation": actual["prediction"] == actual["label"],\n                "Matches saved prediction": actual["prediction"] == recorded["prediction"],\n                "Probability replay within 1e-5": math.isclose(actual["probability"], recorded["probability"], abs_tol=1e-5, rel_tol=1e-5)})\n        replay_status = "EXECUTED"\n        replay_reason = "Read-only local CPU replay of selected validation rows; no selection changes."\n        table([{"Model": actual_config["model"]["pretrained_id"],\n                "Pinned base revision": actual_config["model"]["revision"],\n                "Checkpoint SHA-256": actual_selection["checkpoint_sha256"],\n                "Head": model.config.id2label, "Maximum tokens": actual_config["model"]["max_length"],\n                "Training configuration (recorded, not rerun)": actual_config["training"]}])\n        del model, tokenizer\n    except Exception as error:\n        replay_status = "ERROR"\n        replay_error = {"type": type(error).__name__, "message": str(error)}\n        replay_reason = "Replay stopped; repair the local bundle/environment without weakening its guards."\ntable([{"Replay status": replay_status, "Observation": replay_reason, "Error": replay_error}])\nevidence.append({"Area": "Real model inference", "Experiment": "Private validation checkpoint replay",\n                 "Actual observation": f"{replay_status}: {replay_reason}"})'}, {'id': '75a93080', 'source': 'if replay_report is None:\n    display(Markdown("**Actual V2 model outputs: unavailable in this execution.** "\n                     "The populated utility experiments above are not substitute model predictions."))\nelse:\n    for number, (input_row, values) in enumerate(zip(replay_rows, replay_results), 1):\n        display(Markdown(f"### Model example {number} — `{input_row[\'pair_id\']}`"))\n        display(Markdown("**Input**"))\n        table([{"Premise": input_row["premise"], "Hypothesis": input_row["hypothesis"]}])\n        display(Markdown("**Expected:** the provisional annotation below; saved V2 output is a separate replay reference."\n                         "\\n\\n**Run V2:** `_infer` and `_prediction_report` executed in the preceding cell."\n                         "\\n\\n**Actual Output**"))\n        table([values])\n        correct = values["Matches annotation"]\n        reproducible = values["Matches saved prediction"] and values["Probability replay within 1e-5"]\n        display(Markdown("**Observation:** " + ("matches" if correct else "does not match")\n                         + " the provisional annotation; " + ("reproduces" if reproducible else "differs from")\n                         + " the saved V2 prediction. Confidence remains advisory."))'}, {'id': 'a7573e45', 'source': 'table(before_after)\nmodel_comparison = []\nif replay_report is not None:\n    original_file = run / "comparisons/validation_original.json"\n    if original_file.is_file():\n        original_report = json.loads(original_file.read_text())\n        original_by_id = {r["pair_id"]: r for r in original_report["predictions"]}\n        for actual in replay_report["predictions"]:\n            before = original_by_id.get(actual["pair_id"])\n            if before is None or any(before[k] != actual[k] for k in ("premise", "hypothesis", "label")):\n                raise ValueError("Original comparator does not match the verified input/label")\n            before_correct = before["prediction"] == actual["label"]\n            now_correct = actual["prediction"] == actual["label"]\n            difference = ("Improvement on provisional label" if not before_correct and now_correct else\n                          "Regression on provisional label" if before_correct and not now_correct else\n                          "Both match provisional label" if now_correct else "Both disagree with provisional label")\n            model_comparison.append({"Pair ID": actual["pair_id"], "Expected annotation": actual["label"],\n                "Before (saved original)": before["prediction"], "Before probability": before["probability"],\n                "Before threshold": original_report["threshold"], "Before logits": before.get("raw_logits", "Not recorded"),\n                "V2 (fresh replay)": actual["prediction"], "V2 probability": actual["probability"],\n                "V2 threshold": replay_report["threshold"], "V2 logits": actual["raw_logits"], "Difference": difference})\n        table(model_comparison)\n    else:\n        display(Markdown("**Original comparison NOT AVAILABLE:** saved validation_original.json is missing. "\n                         "No historical output is inferred from V2."))\nelse:\n    table([{"Model before/after": "NOT AVAILABLE in this execution",\n            "Reason": "Private original/V2 validation artifacts and checkpoint are not available",\n            "Claimed predictive improvement": "None from this notebook execution"}])'}, {'id': '66c80512', 'source': 'table(evidence, ["Area", "Experiment", "Actual observation"])'}, {'id': '2aea4d69', 'source': 'observed_utilities = sum(e["Area"] != "Real model inference" for e in evidence)\ndisplay(Markdown(f"**This execution:** {observed_utilities} evidence entries from historical/utility demonstrations; "\n                 f"real checkpoint replay **{replay_status}**. "\n                 + ("Inspect the per-example annotation and replay comparisons above."\n                    if replay_status == "EXECUTED" else\n                    "No fresh trained-model predictions or predictive improvement are claimed.")))\nscratch.cleanup()\nprint("Disposable fixtures removed. Repository source and private run artifacts were not modified.")'}]
ORIGINAL_NOTES = [{'id': '6a8f0919', 'source': '# V2 Model Reliability & Controlled Evaluation — Experimental Demonstration\n\n**PRELIMINARY / EXPERIMENTAL.** A short mentor walkthrough of the current V2 dependency-classification pipeline and its execution controls.\n\n**Problem → desired behavior → current approach → experiments → evidence → limitations.**\n\nThree evidence types are kept separate: **recorded historical response**, **live V2 utility executions with synthetic fixtures**, and **optional real checkpoint replay**. Synthetic probabilities are never presented as model predictions.\n\nSource reference: `Its-soul/cap-stone`, GitHub revision `0548f0cf1bff98a518781d706c9780d290728cab`, with the existing V2 test setup. No production code or V3 changes are required.\n\n*Output provenance: cells executed with in-process IPython because this environment disallows Jupyter kernel socket binding. The real-model section was NOT RUN: no private V2 bundle was configured.*'}, {'id': 'e0f2015e', 'source': "## Problem\n\nThe project tracks which evidence supports an artifact, invalidates affected certificates when evidence changes, and preserves independent valid work during recovery. The deterministic certificate engine remains the authority. V2 supplies **advisory binary dependency predictions**.\n\nThree practical risks matter: a confident semantic prediction can be wrong; changed data or weights can make comparisons invalid; and out-of-order or failed execution can leave misleading results. V2 addresses the experiment-control risks and tests a broader training distribution. Its predictive improvement must be measured separately.\n\n**Important target limitation:** the reviewed target is declared transitive support membership for a named artifact. V2's synthetic training protocol describes closed-function sensitivity. These are not equivalent for zero-effect reads, redundant evidence, or cancellation. This notebook does not claim V2 has resolved that gap."}, {'id': '72bcd51b', 'source': "### Setup\n\nOpen this notebook from the repository root, or set `CERT_RECOVERY_REPO` to your checkout. Use a Jupyter kernel with the project's existing test dependencies (`pytest`, `PyYAML`). Real replay additionally needs the **existing private model environment**, its local V2 bundle, and `CERT_RECOVERY_V2_RUN`.\n\nThis notebook performs no training, downloads, remote requests, or V3 execution. Fixture writes go only to a disposable temporary directory. Private replay reads validation examples and preserves the frozen threshold; it does not select a new model or evaluate a new final set."}, {'id': 'ef9097d2', 'source': "## Existing Behavior\n\n**Input:** “A man is eating pizza.” / “The man owns a bicycle.”\n\n**Expected:** NLI **neutral**; eating pizza does not determine bicycle ownership.\n\n**Actual below:** load the genuine recorded Space response from the repository fixture. This is historical evidence, not fresh inference. The historical service's loaded weights/revision were unknown.\n\n**Why it matters:** successful transport and high reported confidence do not establish semantic correctness. NLI is a different task from binary support membership; this example is **not** an original-vs-V2 dependency comparison."}, {'id': '7445c77b', 'source': "## V2 Goal\n\n**EXPECTED / THEORETICAL — goals, not measured improvements.**\n\n| Scenario | Problematic behavior | Desired V2 behavior |\n|---|---|---|\n| Complete, scoped input | Lose negation, values, or the named target | Preserve the complete model-visible pair |\n| Dependency decision | Treat confidence or NLI as proof | Use the frozen binary threshold; keep advice separate from certificate validity |\n| Controlled comparison | Change data, source, or selection between runs | Reject artifacts that differ from frozen hashes |\n| Stage execution | Run comparisons too early or overwrite failure evidence | Enforce prerequisites and retain each attempt's status |\n| Structural reasoning | Learn identifiers or phrasing instead of the declared support relation | Generalize under controlled role changes; this requires further evidence |"}, {'id': '760b8995', 'source': '## V2 Approach\n\nThe model offers advice; independent certificate rules make the final decision. The guardrails apply throughout the run, rather than only after prediction.\n\nThe diagram describes the complete workflow. In this notebook, historical evidence and synthetic utility checks are available; real model replay still requires the private checkpoint.'}, {'id': 'd0ee6f5b', 'source': '<details>\n<summary>Implementation notes (optional)</summary>\n\n**Frozen protocol + data + source hashes** → **verified splits** → **complete premise/hypothesis tokenization** → **DeBERTa binary head** → **softmax probability + frozen threshold** → **prediction report + saved comparison**.\n\nThe stage controller wraps execution: **training → proxy selection → comparison**, with prerequisite checks and preserved attempt records.\n\n- **Cleaning / splits:** NFC Unicode and whitespace normalization; reject malformed or conflicting rows and split leakage.\n- **Model:** `cross-encoder/nli-deberta-v3-small`, with a fresh two-class dependency head in V2 training. Exact revision, limits and runtime choices come from the private frozen run, not the public default configuration.\n- **Selection:** validation chooses the checkpoint and threshold; final predictions do not choose them.\n- **Reporting:** `_infer` captures logits and all class probabilities; `_prediction_report` applies `p >= threshold`, records correctness and class support.\n- **Integrity / authority:** `verify_frozen` checks the run; `execute_stage` controls attempts. Model advice cannot replace the independent deterministic verifier.\n\nWe reuse these functions and the existing `frozen_run` fixture from `tests/test_v2_execution.py`. The fixture contains six explicitly synthetic rows and real source hashes, **no checkpoint or trained model**.\n\n</details>'}, {'id': 'd2ff1306', 'source': '## Test Cases\n\n### Test Case 1 — Preserve a complete scoped input\n\n**Input:** a synthetic pair with decomposed Unicode, repeated whitespace, an explicit Q-only scope, negation, and the value 17.\n\n**Before / problem:** the raw string has inconsistent formatting. Losing the scope or “not” during cleanup would change the task.\n\n**Expected:** canonical Unicode and single spaces; preserve `Q-only`, `not`, `17`, and `café`. This tests text handling, not a learned dependency prediction.\n\n**Run V2 / Actual Output:**'}, {'id': '141c7c00', 'source': '### Test Case 2 — A positive prediction can have probability below 0.5\n\n**Input:** an explicitly **synthetic** dependency probability of 0.30, with illustrative frozen threshold 0.20 and provisional label 1.\n\n**Before / problem:** applying a conventional 0.50 cutoff gives class 0. It is not the V2 frozen decision rule in this illustration.\n\n**Expected:** V2 predicts class 1 because `0.30 >= 0.20`. Its reported predicted-label probability remains 0.30; this is not a high-confidence prediction.\n\n**Run V2 / Actual Output:** the real reporting function; **no model forward pass and no historical baseline**.'}, {'id': 'c0a6179a', 'source': '### Test Case 3 — Reject changed validation evidence\n\n**Input:** a valid bundle from the existing synthetic fixture, followed by a changed validation premise in its disposable copy.\n\n**Before / problem:** the input would differ from the frozen dataset. Silently evaluating it would compromise comparability.\n\n**Expected:** accept the untouched fixture; reject the altered split. No historical implementation is run or alleged to lack this protection.\n\n**Run V2 / Actual Output:**'}, {'id': '7ae583b0', 'source': '## V2 Reliability Demonstrations\n\n### Stage ordering\n\n**Input / problem:** request comparison before training or proxy selection.\n\n**Expected:** V2 rejects it before creating an attempt. **Run V2 / Actual Output:**'}, {'id': '93dc75fd', 'source': '### Failure preservation and CPU-only enforcement\n\n**Controlled fault injection — not real training.** Reuse the existing test strategy: substitute only the external training callback and a tiny hardware adapter, then execute the **real V2 stage controller** on a disposable fixture.\n\n**Input / problem:** a training backend raises an error, or the simulated hardware reports a GPU despite this protocol being CPU-only.\n\n**Expected:** a persisted `ERROR` record in both the attempt and run manifest. A second attempt on the same failed stage must be rejected. These notebook-local substitutions are restored immediately.'}, {'id': '3d2f2fd7', 'source': '## Real V2 Checkpoint Replay\n\n**Configure:** set `CERT_RECOVERY_V2_RUN` to your existing private V2 run folder before starting this notebook, and select its existing model-environment kernel. Rerun all cells.\n\nWe follow `tests/integration/test_v2_checkpoint.py`: `verify_frozen` → checkpoint file hashes → local-only tokenizer/model loading → verified validation rows → `_infer` → `_prediction_report` → saved-prediction comparison. No checkpoint is created and no threshold is retuned.\n\nChoose up to three **validation** rows from the saved report: one correct dependent case, one correct independent case, and the highest-confidence error if present. This is a deliberately selected qualitative demonstration, **not an unbiased accuracy estimate**. Annotation labels are provisional expectations; agreement with a saved prediction checks reproducibility, not truth.\n\nIf the private bundle is missing, the section displays **NOT RUN** and the notebook remains executable. An available but invalid bundle displays **ERROR**; it is not bypassed or replaced with synthetic inference.'}, {'id': 'bc6caeb8', 'source': '## Before vs V2\n\n**Utility experiments:** compare raw/valid/simulated conditions with the actual guarded behavior. The 0.50 rule is illustrative, not a historical model.\n\n**Model comparison:** when available, read the original and V2 saved validation reports for the **same verified inputs**. Saved artifacts are not fresh original inference. Without private artifacts, no numerical original-vs-V2 improvement is claimed.'}, {'id': '6c83ef81', 'source': '## Gap analysis and improvements\n\n| Observed or unresolved issue | Practical improvement |\n|---|---|\n| Recorded NLI case is wrong at 99.57% confidence | Use reviewed task-specific labels; assess calibration on adequate data; keep confidence advisory |\n| Cleaning and integrity pass, but these do not establish structural reasoning | Review target scope and exhaustivity; evaluate valid renames and role changes separately from ordinary F1 |\n| V2 protocol uses provisional function-sensitivity labels | Reconcile labels with declared transitive support membership; do not silently relabel historical results |\n| Real checkpoint absent in this environment | Replay the existing verified local V2 checkpoint and inspect the selected failures |\n| Simulated failures were recorded correctly | Test operational failures in an authorized run; retain artifacts and investigate rather than auto-retry |\n\n**Fallback:** when real replay cannot run, display `NOT RUN` or `ERROR`. There is no substitute heuristic prediction or invented checkpoint. The deterministic certificate verifier remains independent of semantic advice.\n\nCandidate V3 remains disabled and outside this notebook. Human annotation review and experiment authorization remain separate prerequisites.'}, {'id': 'db599503', 'source': '## Evidence\n\nOnly observations produced in this execution are included below. Hardware/backend simulations and synthetic probability inputs are labeled. No aggregate model accuracy is estimated from these selected examples.'}, {'id': '253a9566', 'source': "## Conclusion\n\n**Problem:** confidence alone cannot establish a correct dependency decision, and uncontrolled changes can undermine experimental evidence.\n\n**What V2 provides:**\n- Complete-input cleaning and split validation.\n- Frozen checkpoint/threshold selection and prediction reporting.\n- Hash-bound artifact verification.\n- Ordered execution and preserved attempt failures.\n\n**Experiments:** the populated cells show what these utilities actually did. The private replay's observed status below determines whether this notebook also demonstrates trained-model behavior.\n\n**Limitations:** synthetic reliability fixtures do not demonstrate model accuracy; historical NLI is a separate task; annotations remain provisional; selected validation examples do not establish generalization or calibration. No V3 approval, training or evaluation is implied."}]
ORIGINAL_DIAGRAM = 'from IPython.display import HTML, display\n\ndisplay(HTML(\'<style>\\n#v2-reliability-story { --foreground:#202d40; --background:#ffffff; --muted:#edf1f5; --muted-foreground:#566579; --viz-series-1:#2864b5; --viz-series-2:#71549b; font:15px/1.5 system-ui,sans-serif; max-width:900px; margin:15px auto; }\\n#v2-reliability-story .text-small {font-size:13px;}\\n#v2-reliability-story h2 {font-size:22px;font-weight:500;line-height:1.35;margin:0;}\\n#v2-reliability-story h3 {font-size:16px;font-weight:500;}\\n#v2-reliability-story hr {border:0;border-top:1px solid #dce2ea;margin:16px 0;}\\n</style><div id="v2-reliability-story" role="figure" aria-label="Six-step V2 advisory prediction workflow, surrounded by reliability safeguards. The deterministic certificate engine alone decides certificate validity. Historical evidence, synthetic utility tests and optional checkpoint replay are separate evidence types.">\\n  <style>\\n    #v2-reliability-story { color:var(--foreground); font-family:inherit; }\\n    #v2-reliability-story .story-layout { display:grid; grid-template-columns:minmax(0,1.5fr) minmax(0,1fr); gap:38px; margin-top:24px; }\\n    #v2-reliability-story .flow { list-style:none; padding:0; margin:0; }\\n    #v2-reliability-story .stage { display:grid; grid-template-columns:36px minmax(0,1fr); gap:16px; position:relative; padding-bottom:27px; }\\n    #v2-reliability-story .stage:last-child { padding-bottom:0; }\\n    #v2-reliability-story .step-number { display:flex; align-items:center; justify-content:center; width:36px; height:36px; border-radius:50%; background:var(--muted); color:var(--foreground); font-weight:500; }\\n    #v2-reliability-story .stage:not(:last-child)::after { content:"↓"; position:absolute; left:0; top:43px; width:36px; text-align:center; color:var(--muted-foreground); }\\n    #v2-reliability-story .stage-name { display:block; font-weight:500; margin:0 0 5px; }\\n    #v2-reliability-story .stage-detail { display:block; }\\n    #v2-reliability-story .advisory .step-number { background:color-mix(in srgb,var(--viz-series-1) 16%,var(--background)); }\\n    #v2-reliability-story .advisory-label { display:inline-flex; align-items:center; gap:7px; margin-top:7px; }\\n    #v2-reliability-story .dot { display:inline-block; width:9px; height:9px; border-radius:50%; background:var(--viz-series-1); flex-shrink:0; }\\n    #v2-reliability-story .authority-dot { background:var(--viz-series-2); }\\n    #v2-reliability-story .guards { list-style:none; margin:18px 0 0; padding:0; }\\n    #v2-reliability-story .guards li { display:flex; flex-wrap:wrap; gap:5px; margin-bottom:15px; }\\n    #v2-reliability-story .guard-result { font-weight:500; }\\n    #v2-reliability-story .side h3 { margin:0 0 7px; }\\n    #v2-reliability-story .authority { margin-top:38px; }\\n    #v2-reliability-story .authority-name { display:flex; align-items:center; gap:8px; font-weight:500; margin:14px 0 8px; }\\n    #v2-reliability-story .authority p { margin:8px 0 0; }\\n    #v2-reliability-story .evidence-types { display:grid; grid-template-columns:repeat(3,minmax(0,1fr)); gap:22px; margin-top:16px; }\\n    #v2-reliability-story .evidence-types strong { display:block; font-weight:500; margin-bottom:6px; }\\n    #v2-reliability-story .evidence-types span { display:block; margin-top:4px; }\\n    #v2-reliability-story .evidence-heading { margin-top:25px; }\\n    @media(max-width:540px) {\\n      #v2-reliability-story .story-layout { grid-template-columns:1fr; gap:26px; }\\n      #v2-reliability-story .side { display:grid; grid-template-columns:1fr; }\\n      #v2-reliability-story .authority { margin-top:23px; }\\n      #v2-reliability-story .evidence-types { grid-template-columns:1fr; gap:15px; }\\n    }\\n  </style>\\n  <h2>Predict with V2. Keep the final decision separate.</h2>\\n  <div class="story-layout">\\n    <ol class="flow" aria-label="Main advisory prediction workflow">\\n      <li class="stage"><span class="step-number" aria-hidden="true">1</span><div><span class="stage-name">Input</span><span class="stage-detail text-small">Complete premise + hypothesis<br>Which source supports the target?</span></div></li>\\n      <li class="stage"><span class="step-number" aria-hidden="true">2</span><div><span class="stage-name">Prepare &amp; Verify</span><span class="stage-detail text-small">Clean the text; confirm data, splits,<br>settings and artifacts are unchanged.</span></div></li>\\n      <li class="stage advisory"><span class="step-number" aria-hidden="true">3</span><div><span class="stage-name">V2 Model · Analyze</span><span class="stage-detail text-small">DeBERTa estimates the probability<br>of a dependency.</span><span class="advisory-label text-small"><span class="dot" aria-hidden="true"></span>Model prediction · advisory only</span></div></li>\\n      <li class="stage advisory"><span class="step-number" aria-hidden="true">4</span><div><span class="stage-name">Decision</span><span class="stage-detail text-small">Use the frozen threshold to predict<br>“depends on” or “independent.”</span></div></li>\\n      <li class="stage"><span class="step-number" aria-hidden="true">5</span><div><span class="stage-name">Reliability Checks · Control</span><span class="stage-detail text-small">Check order, detect invalid changes,<br>and preserve failures throughout the run.</span></div></li>\\n      <li class="stage"><span class="step-number" aria-hidden="true">6</span><div><span class="stage-name">Evidence / Output · Record</span><span class="stage-detail text-small">Prediction, probability, comparisons,<br>and experiment validity / status.</span></div></li>\\n    </ol>\\n    <aside class="side" aria-label="Reliability safeguards and separate certificate authority">\\n      <section>\\n        <h3>Reliability controls</h3>\\n        <div class="text-small">Guardrails around the whole run</div>\\n        <ul class="guards">\\n          <li><span>Changed data</span><span aria-hidden="true">→</span><span class="guard-result">Block</span></li>\\n          <li><span>Wrong order</span><span aria-hidden="true">→</span><span class="guard-result">Block</span></li>\\n          <li><span>Training failure</span><span aria-hidden="true">→</span><span class="guard-result">Record ERROR</span></li>\\n          <li><span>Invalid hardware</span><span aria-hidden="true">→</span><span class="guard-result">Block</span></li>\\n          <li><span>Valid run</span><span aria-hidden="true">→</span><span class="guard-result">Continue</span></li>\\n        </ul>\\n      </section>\\n      <section class="authority" aria-label="Final certificate authority, independent of the model">\\n        <hr>\\n        <h3>Final authority</h3>\\n        <div class="authority-name"><span class="dot authority-dot" aria-hidden="true"></span><span>Deterministic<br>Certificate Engine</span></div>\\n        <p class="text-small">Its independent rules decide<br>certificate validity and recovery.</p>\\n        <p class="text-small">Model predictions never approve<br>or invalidate a certificate.</p>\\n      </section>\\n    </aside>\\n  </div>\\n  <div class="evidence-heading"><hr><h3>Three separate kinds of evidence</h3></div>\\n  <div class="evidence-types" aria-label="Evidence types and actual notebook execution status">\\n    <div><strong>Historical evidence</strong><span class="text-small">Previously recorded result</span><span class="text-small">Shown in this notebook</span></div>\\n    <div><strong>Live utility tests</strong><span class="text-small">Controlled synthetic checks</span><span class="text-small">Executed · not model predictions</span></div>\\n    <div><strong>Real V2 replay</strong><span class="text-small">Actual private checkpoint</span><span class="text-small">Optional · NOT RUN here</span></div>\\n  </div>\\n</div>\\n\'))'
experiment_evidence = []
for block in EXPERIMENT_CELLS:
    with capture_output(stdout=True, stderr=True, display=True) as captured:
        exec(compile(block["source"], "V2 experiment " + block["id"], "exec"), globals())
    experiment_evidence.append({**block, "stdout": captured.stdout, "stderr": captured.stderr,
                                "outputs": captured.outputs})


In [2]:

def esc(value):
    return html.escape(str(value))

main_style = """<style>
#v2-demo {font:15px/1.55 system-ui,sans-serif;color:#243246;max-width:880px;margin:auto;}
#v2-demo h2 {font-size:21px;font-weight:600;margin:28px 0 12px;}
#v2-demo p {margin:8px 0;}
#v2-demo .small {font-size:13px;color:#536174;}
#v2-demo .result {display:grid;grid-template-columns:1fr 1fr;gap:22px;margin:15px 0;}
#v2-demo .answer {font-size:21px;font-weight:600;display:block;margin-top:3px;}
#v2-demo .flow {margin:16px 0;font-weight:600;word-spacing:4px;}
#v2-demo .authority {margin:12px 0;font-size:14px;}
#v2-demo table {width:100%;border-collapse:collapse;margin:12px 0;}
#v2-demo td,#v2-demo th {text-align:left;padding:10px 8px;border-bottom:1px solid #e1e7ed;vertical-align:top;}
#v2-demo th {font-weight:600;font-size:13px;color:#536174;}
#v2-demo .pass {color:#226747;font-weight:600;}
#v2-demo .warn {color:#805509;font-weight:600;}
@media(max-width:500px){#v2-demo .result{grid-template-columns:1fr;gap:10px;}#v2-demo td,#v2-demo th{padding:8px 4px;}}
</style>"""

historical_html = f"""
<h2>The problem: confidence can be misleading</h2>
<p>“{esc(record['input']['premise'])}”<br>“{esc(record['input']['hypothesis'])}”</p>
<div class="result"><div><span class="small">Expected</span><span class="answer">{esc(record['expected_label'].capitalize())}</span></div>
<div><span class="small">Actual recorded answer</span><span class="answer">{esc(normal['label'].capitalize())} · {esc(normal['confidence_text'])}</span></div></div>
<p class="small">Historical NLI example. Eating pizza tells us nothing about bicycle ownership.</p>
<h2>V2: predict, then keep decisions controlled</h2>
<div class="flow">Input → Verify → Model → Prediction → Checks → Output</div>
<p class="authority"><strong>V2:</strong> advisory dependency prediction.<br>
<strong>Certificate engine:</strong> independent rules make the final certificate decision.</p>
"""

checks = [
    ("Messy input", "Keep scope, “not”, 17 and Café", "Preserved"),
    ("Changed validation data", "Reject the changed input", "Blocked"),
    ("Wrong execution order", "Stop before comparison", "Blocked"),
    ("Failed training backend", "Retain the failure evidence", "ERROR saved"),
]
assert accepted['premise'] == expected_text
assert integrity_error['type'] == 'ValueError'
assert ordering_error['message'] == 'Successful selected training required'
assert persisted == failure_state and failure_state['status'] == 'ERROR'
check_rows = ''.join(f'<tr><td>{esc(a)}</td><td>{esc(b)}</td><td class="pass">{esc(c)}</td></tr>' for a,b,c in checks)
checks_html = '<h2>Four actual reliability examples</h2><p class="small">Controlled synthetic checks — not model predictions.</p><table><tr><th>Example</th><th>Expected</th><th>Actual</th></tr>'+check_rows+'</table>'

model_html = ''
if replay_report is not None:
    model_html = '<h2>Actual V2 predictions</h2>'
    for row, result in zip(replay_rows, replay_results):
        model_html += ('<p>'+esc(row['premise'])+'<br>'+esc(row['hypothesis'])+'</p>'
                       '<p><strong>Expected:</strong> '+('Depends on' if row['label'] else 'Independent')
                       +' · <strong>Actual:</strong> '+('Depends on' if result['Actual V2 prediction'] else 'Independent')
                       +' · <strong>Dependency probability:</strong> '+f"{result['P(depends_on)']:.1%}"+'</p>')
    model_html += '<p class="small">Selected validation examples; labels are provisional.</p>'

model_state = {'NOT RUN':'⚠ Not run', 'ERROR':'⚠ Error', 'EXECUTED':'✓ Executed'}[replay_status]
statuses = [("Input handling", "✓ Passed"), ("Integrity checks", "✓ Passed"),
            ("Execution controls", "✓ Passed"), ("Failure handling", "✓ Passed"),
            ("Real V2 replay", model_state), ("Measured V2 improvement", "⚠ Not claimed")]
status_rows = ''.join('<tr><td>'+esc(a)+'</td><td class="'+('warn' if b.startswith('⚠') else 'pass')+'">'+esc(b)+'</td></tr>' for a,b in statuses)
reason = ('Private model bundle unavailable in this environment.' if replay_status == 'NOT RUN' else
          'Replay stopped with an error; see technical evidence.' if replay_status == 'ERROR' else
          'Selected examples check replay; they do not establish general improvement.')
status_html = '<h2>What we can conclude</h2><table><tr><th>Item</th><th>Status</th></tr>'+status_rows+'</table><p class="small">'+reason+'</p>'
next_step = '<p><strong>Next:</strong> '+('Review the replay failures and annotation scope.' if replay_status == 'EXECUTED' else 'Replay the existing private V2 checkpoint, then inspect its errors.')+'</p>'
display(HTML(main_style+'<div id="v2-demo">'+historical_html+checks_html+model_html+status_html+next_step+'</div>'))


In [3]:

def evidence_markup(data):
    if 'text/html' in data:
        return data['text/html']
    value = data.get('text/markdown', data.get('text/plain', ''))
    return '<pre class="note">'+html.escape(str(value))+'</pre>'

details_style = """<style>
#v2-details {font:14px/1.5 system-ui,sans-serif;max-width:880px;margin:28px auto;color:#243246;}
#v2-details details {margin:14px 0;}
#v2-details summary {font-weight:600;cursor:pointer;padding:8px 0;}
#v2-details pre {white-space:pre-wrap;overflow-wrap:anywhere;font-size:12px;padding:12px;background:#f3f5f8;color:#243246;}
#v2-details .note {font-family:system-ui,sans-serif;background:transparent;padding:4px 0;font-size:14px;}
#v2-details .details-body {padding:4px 0 4px 16px;}
</style>"""
setup = """<details><summary>Setup &amp; real checkpoint replay</summary><div class="details-body">
<p>Open from the repository root, or set <code>CERT_RECOVERY_REPO</code>. Use a kernel with the existing project/test dependencies.</p>
<p>For model replay, set <code>CERT_RECOVERY_V2_RUN</code> to the existing private V2 run and select its model-environment kernel. Then run all cells.</p>
<p>Replay uses local verified weights and validation examples. It does not train, retune, download weights, or execute V3.</p></div></details>"""
titles = ['Environment and helpers','Historical recorded response','Existing synthetic fixture',
          'Input cleaning','Threshold decision and invalid probability','Artifact tampering',
          'Stage ordering','Failure preservation and simulated hardware','Private checkpoint replay',
          'Per-example replay comparisons','Original versus V2 comparison','Evidence summary','Cleanup']
experiments = '<details><summary>Experiment evidence · exact inputs, code and outputs</summary><div class="details-body">'
for title, item in zip(titles, experiment_evidence):
    experiments += '<details><summary>'+esc(title)+'</summary><div class="details-body">'
    experiments += '<pre><code>'+esc(item['source'])+'</code></pre>'
    if item['stdout']: experiments += '<pre>'+esc(item['stdout'])+'</pre>'
    if item['stderr']: experiments += '<pre>'+esc(item['stderr'])+'</pre>'
    for output in item['outputs']:
        experiments += evidence_markup(output.data)
    experiments += '</div></details>'
experiments += '</div></details>'
protocol = '<details><summary>Protocol, limitations &amp; original notes</summary><div class="details-body">'
for note in ORIGINAL_NOTES:
    protocol += '<pre class="note">'+esc(note['source'])+'</pre>'
protocol += '</div></details>'
diagram = '<details><summary>Detailed workflow diagram</summary><div class="details-body">'
with capture_output(stdout=True,stderr=True,display=True) as diagram_capture:
    exec(ORIGINAL_DIAGRAM,globals())
for output in diagram_capture.outputs:
    diagram += evidence_markup(output.data)
diagram += '</div></details>'
display(HTML(details_style+'<div id="v2-details"><h2>Technical Details</h2>'+setup+experiments+protocol+diagram+'</div>'))
